# Create and Upload I14Y Concept

At the moment, the strategy is the following:

- run the pipeline daily
- create the codelist entries
- get the id of the most current concept via the identifier political-parties
- check if the codelist entries differ from the existing concept on I14Y, if not --> end, else:
- set the validTo value of the existing concept to the current date
- create a new version of the concept (with the same identifier political-parties and the version number as current date (e.g. 2026.10.2))
- set the new concept to public

## Import Libraries

In [1]:
import os
import json
import pandas as pd
import requests
from dotenv import load_dotenv
from datetime import datetime

## Information

The concepts on I14Y have two different identifiers:

- static identifier: `identifiers` (here: political-parties)
- versioned identifier: `conceptId` (e.g. 01a120af-e345-72be-888a-4e1ebb918106 --> this is a UUID)

## Define Functions

In [2]:
def create_concept_entries_from_json(parties):
    """
    Create concept entries from the parties JSON data.

    Args:
        parties (dict): The parties JSON data.

    Returns:
        dict: The new concept entries.
    """
    
    # base für codes json
    entries = {
        "data": [],
    }

    for id, party in parties.items():

        # only identity is relevant
        party = party.get("identity")

        if party.get("name") and pd.isna(party.get("dissolution_date")):  # check if the party has a name and is not deprecated
            code_entry = {
                "annotations": [
                    {
                        "type": "EXT_RESOURCE",
                        "uri": f"https://politics.ld.admin.ch/political-party/{id}"
                    }
                ],
                "code": str(id),  # get the id of the party
                "name": party.get("name")
            }
            # if the party has a parent, add the parent code to the code entry
            child_of = [relation for relation in party.get("relationships", []) if relation.get("type_label") == "child-of"]
            if len(child_of) > 0:
                child_of = child_of[0]  # get the first child_of relation
                code_entry["parentCode"] = str(child_of.get("party_to"))
            entries["data"].append(code_entry)

    return entries

In [3]:
def get_current_concept(static_identifier):
    """
    Get the latest concept from the i14y API.

    Args:
        static_identifier (str): The static identifier of the concept that never changes, e.g. "political-parties".

    Returns:
        dict: The latest concept.
    """
    
    url = f"https://api-a.i14y.admin.ch/api/public/v1/concepts/"

    params = {
        'conceptIdentifier': static_identifier, 
    }

    try:
        response = requests.get(url, params=params, verify = False, timeout=10)
        response.raise_for_status()  # Raise an exception for HTTP errors
        current_concept = response.json()
    except requests.exceptions.HTTPError as e:
        print(f"HTTP-Error {e.response.status_code}: {e.response.text[:200]}")
    except requests.exceptions.Timeout:
        print("Timeout")

    print(f"Current concept: {current_concept}")


    # sort data by 'version' in descending order
    current_concept = sorted(current_concept["data"], key=lambda x: x['version'], reverse=True)
    current_concept = {"data": current_concept[0]}

    return current_concept

In [ ]:
def get_current_concept_entries(versioned_identifier):
    """
    Get the latest concept entries from the i14y API.

    Args:
        versioned_identifier (str): The versioned identifier of the concept, e.g. "01a120af-e345-72be-888a-4e1ebb918106".


    Returns:
        dict: The latest concept entries.
    """
    
    url = f"https://api-a.i14y.admin.ch/api/public/v1/concepts/{versioned_identifier}/codelist-entries/exports/json"

    response = requests.get(url, verify = False)

    if response.status_code == 200:
        current_concept_entries = json.loads(response.content)
    else:
        print(f"Failed to get data, status code: {response.status_code}")

    return current_concept_entries

In [5]:
def compare_concept_entries(current_concept_entries, new_concept_entries):
    """
    Compare the current concept entries with the new concept entries.

    Args:
        current_concept_entries (dict): The current concept entries.
        new_concept_entries (dict): The new concept entries.

    Returns:
        bool: True if there are changes, False otherwise.
    """
    
    current_codes = set(json.dumps(d, sort_keys=True) for d in current_concept_entries["data"])
    new_codes = set(json.dumps(d, sort_keys=True) for d in new_concept_entries["data"])

    if current_codes == new_codes:
        print("No changes detected. The code list is up to date.")
        return False
    else:
        print("Changes detected. The code list needs to be updated.")
        return True

In [6]:
def get_access_token(client_id, client_secret):
    """
    Get an access token from the i14y identity provider.

    Args:
        client_id (str): The client ID.
        client_secret (str): The client secret.
    
    Returns:
        str: The access token.
    """
    data = {'grant_type': 'client_credentials'}
    response = requests.post("https://identity.i14y.a.c.bfs.admin.ch/realms/bfs-sis-a/protocol/openid-connect/token", data=data, verify=False, auth=(client_id, client_secret))
    data = response.json()
    return 'Bearer ' + data['access_token']

In [7]:
def set_concept_end_date(current_concept, concept_template, end_date, access_token):
    """
    Set the end date of a concept in the i14y API.

    Args:
        current_concept (dict): The current concept.
        concept_template (dict): The concept template.
        end_date (str): The end date in isoformat, e.g. "2024-06-30T00:00:00".
        access_token (str): The access token.
    
    Returns:
        dict: The response from the API.
    """

    headers = {
        'Content-Type': 'application/json',  # To make a valid request it's necessary to state the correct 'Content-Type' (Note: in the other POST APIs [point 6 and 7: import codelist entries] there is not need to state the 'Content-Type', in fact stating it it could lead to errors)
        'User-Agent': "Political Parties (contact: benedikt.hitz@bfh.ch)",
        'Authorization': access_token
    }

    url = f'https://api-a.i14y.admin.ch/api/partner/v1/concepts/{current_concept["data"]["id"]}'

    concept_template["data"]["validFrom"] = current_concept["data"]["validFrom"]
    concept_template["data"]["validTo"] = end_date
    concept_template["data"]["version"] = current_concept["data"]["version"]

    response = requests.put(url, headers=headers, data=json.dumps(concept_template), verify=False)

    print(f'Status-Code: {response.status_code} - {response.text}')


In [8]:
def create_concept(concept, valid_from, access_token):
    """
    Create a new concept in the i14y API.

    Args:
        concept (dict): The concept to create.
        valid_from (str): The valid from date in isoformat, e.g. "2024-06-30T00:00:00".
        access_token (str): The access token.

    Returns:
        str: The id of the newly created concept.
    """

    headers = {
        'Content-Type': 'application/json',  # To make a valid request it's necessary to state the correct 'Content-Type' (Note: in the other POST APIs [point 6 and 7: import codelist entries] there is not need to state the 'Content-Type', in fact stating it it could lead to errors)
        'User-Agent': "Political Parties (contact: benedikt.hitz@bfh.ch)",
        'Authorization': access_token
    }

    url = 'https://api-a.i14y.admin.ch/api/partner/v1/concepts'


    concept["data"]["validFrom"] = valid_from
    # parse valid_from to datetime object and format it as "YYYY.MM.DD"
    version_date = datetime.fromisoformat(valid_from)
    version_date = f"{version_date.year}.{version_date.month}.{version_date.day}"
    concept["data"]["version"] = version_date

    # remove the "validTo" field from the concept before sending the POST request
    if "validTo" in concept["data"]:
        del concept["data"]["validTo"]


    response = requests.post(url, headers=headers, data=json.dumps(concept), verify=False)

    if response.status_code == 201:
        print(f'Status-Code: {response.status_code}')
        concept_id = response.json()
        return concept_id
    else:
        print(f"Error: {response.status_code} - {response.text}")

In [9]:
def add_concept_entries(concept_id, concept_entries, access_token):
    """
    Add concept entries to a concept in the i14y API.

    Args:
        concept_id (str): The id of the concept.
        concept_entries (dict): The concept entries to add.
        access_token (str): The access token.
    """

    headers = { 
        'Authorization': access_token,
        'User-Agent': "Political Parties (contact: benedikt.hitz@bfh.ch)",
    }

    url = f'https://api-a.i14y.admin.ch/api/partner/v1/concepts/{concept_id}/codelist-entries/imports/json'

    payload = json.dumps(concept_entries).encode('utf-8')

    files = {
        'file': ('i14_concept_entries.json', payload, 'application/json')
    }
    response = requests.post(url, headers=headers, files=files, verify = False)

    print(f'Status-Code: {response.status_code} - {response.text}')
    print(response)
        

In [10]:
def set_concept_to_public(concept_id, access_token):
    """
    Set a concept to public in the i14y API.

    Args:
        concept_id (str): The id of the concept.
        access_token (str): The access token.
    """

    headers = { 
        'Authorization': access_token,
        'User-Agent': "Political Parties (contact: benedikt.hitz@bfh.ch)",
    }

    params = {
        'conceptId': concept_id,
        'level': "Public" 
    }

    url = f'https://api-a.i14y.admin.ch/api/partner/v1/concepts/{concept_id}/publication-level'


    response = requests.put(url, headers=headers, params=params)

    print(f'Status-Code: {response.status_code} - {response.text}')
    print(response)
    

In [11]:
def delete_concept(concept_id, access_token):
    """
    Delete a concept in the i14y API.

    Args:
        concept_id (str): The id of the concept.
        access_token (str): The access token.
    """

    headers = { 
        'Authorization': access_token,
        'User-Agent': "Political Parties (contact: benedikt.hitz@bfh.ch)",
    }

    url = f"https://api-a.i14y.admin.ch/api/partner/v1/concepts/{concept_id}"

    response = requests.delete(url, headers=headers, verify = False)

    if response.status_code == 204:
        print("Code List entries deleted successfully")
    else:
        print(f"Error: {response.status_code} - {response.text}")

## Run

In [23]:
# read parties.json to dict
with open('../data/output/parties.json', 'r', encoding='utf-8') as file:
    parties = json.load(file)

new_concept_entries = create_concept_entries_from_json(parties)

In [24]:
current_concept = get_current_concept("political-parties")

Current concept: {'data': [{'codeListEntryDefaultSortProperty': 'Position', 'codeListEntryValueMaxLength': 30, 'codeListEntryValueType': 'String', 'conceptType': 'CodeList', 'conformsTo': [], 'description': {'de': 'Alle politischen Parteien der Schweiz auf Stufe Bund und Kantone.', 'en': 'All political parties of Switzerland at federal and cantonal level.', 'fr': 'Tous les partis politiques de Suisse au niveau fédéral et cantonal.', 'it': 'Tutti i partiti politici della Svizzera a livello federale e cantonale.', 'rm': 'Tuts ils partis politics da la Svizra a livel federal e cantunal.'}, 'id': '01a120af-e345-72be-888a-4e1ebb918106', 'identifiers': ['political-parties'], 'isLocked': False, 'keywords': [], 'name': {'de': 'Liste der Identifikatoren politischer Parteien', 'en': 'List of identifiers of political parties', 'fr': 'Liste des identifiants des partis politiques', 'it': 'Elenco degli identificatori dei partiti politici', 'rm': 'Lista dals identificaturs da las partidas politicas'}

/Users/hib1/Nextcloud/Projekte/26_BK_Parties/fch-political-party-pipeline/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api-a.i14y.admin.ch'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [25]:
print(current_concept["data"]["id"])

01a120af-e345-72be-888a-4e1ebb918106


In [26]:
current_concept_entries = get_current_concept_entries(current_concept["data"]["id"])

/Users/hib1/Nextcloud/Projekte/26_BK_Parties/fch-political-party-pipeline/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api-a.i14y.admin.ch'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [27]:
update = compare_concept_entries(current_concept_entries, new_concept_entries)
print(f"Update needed: {update}")

Changes detected. The code list needs to be updated.
Update needed: True


## If Update Necessary

In [28]:
ENV = "INT"

# Load the env file
load_dotenv(f"../.env.{ENV.lower()}")

True

In [29]:
# get the access token
access_token = get_access_token(os.getenv('I14Y_CLIENT_ID'), os.getenv('I14Y_CLIENT_SECRET'))

/Users/hib1/Nextcloud/Projekte/26_BK_Parties/fch-political-party-pipeline/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'identity.i14y.a.c.bfs.admin.ch'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [30]:
# set the end date of the current concept to today
now = datetime.now().isoformat()
with open('../data/output/i14y_concept_template.json', 'r', encoding='utf-8') as file:
    concept_template = json.load(file)
set_concept_end_date(current_concept, concept_template, now, access_token)

/Users/hib1/Nextcloud/Projekte/26_BK_Parties/fch-political-party-pipeline/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api-a.i14y.admin.ch'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status-Code: 204 - 


In [31]:
# create a new concept with the new concept entries
now = datetime.now().isoformat()
with open('../data/output/i14y_concept_template.json', 'r', encoding='utf-8') as file:
    concept_template = json.load(file)
new_concept_id = create_concept(concept_template, now, access_token)

/Users/hib1/Nextcloud/Projekte/26_BK_Parties/fch-political-party-pipeline/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api-a.i14y.admin.ch'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status-Code: 201


In [32]:
# add the new concept entries to the new concept
add_concept_entries(new_concept_id, new_concept_entries, access_token)

/Users/hib1/Nextcloud/Projekte/26_BK_Parties/fch-political-party-pipeline/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api-a.i14y.admin.ch'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status-Code: 204 - 
<Response [204]>


In [33]:
# set the new concept to public
set_concept_to_public(new_concept_id, access_token)

Status-Code: 204 - 
<Response [204]>


## Initial Run

If no concept is existing yet.

In [35]:
# get token

ENV = "INT"

# Load the env file
load_dotenv(f"../.env.{ENV.lower()}")

token = get_access_token(os.getenv('I14Y_CLIENT_ID'), os.getenv('I14Y_CLIENT_SECRET'))

/Users/hib1/Nextcloud/Projekte/26_BK_Parties/fch-political-party-pipeline/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'identity.i14y.a.c.bfs.admin.ch'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [6]:
headers = {
    'Content-Type': 'application/json',  # To make a valid request it's necessary to state the correct 'Content-Type' (Note: in the other POST APIs [point 6 and 7: import codelist entries] there is not need to state the 'Content-Type', in fact stating it it could lead to errors)
    'User-Agent': "Political Parties (contact: benedikt.hitz@bfh.ch)",
    'Authorization': token 
}

file_path = "../data/output/test_concept.json" #state the right file name here
with open(file_path, 'r') as file:
    json_data = json.load(file)

url = 'https://api-a.i14y.admin.ch/api/partner/v1/concepts'

response = requests.post(url, headers=headers, json=json_data, verify = False)
if response.status_code == 201:
    print(f'Status-Code: {response.status_code}')
    print(f'Concept Id: {response.content}')
else:
    print(f"Error: {response.status_code} - {response.text}")

/Users/bene/Nextcloud BFH/Projekte/26_BK_Parties/fch-political-party-pipeline/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api-a.i14y.admin.ch'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status-Code: 201
Concept Id: b'"01a11cae-ee9d-791e-913f-981a28e6a80b"'
